> Adapted from *Dive into Deep Learning*, section [Densely Connected Networks (DenseNet)](https://d2l.ai/chapter_convolutional-modern/densenet.html). Not affiliated with or endorsed by the book's authors.
> Source, authors, licenses and changes: [Attribution and changes](#Attribution-and-changes), at the end of this notebook.

Previous: [Section 8.6: Residual Networks (ResNet) and ResNeXt](08.06_resnet.ipynb) | Next: [Section 8.8: Designing Convolution Network Architectures](08.08_cnn-design.ipynb)

# Densely Connected Networks (DenseNet)

ResNet significantly changed the view of how to parametrize the functions in deep networks. *DenseNet* (dense convolutional network) is to some extent the logical extension of this (Huang et al., 2017).
DenseNet is characterized by both the connectivity pattern where
each layer connects to all the preceding layers
and the concatenation operation (rather than the addition operator in ResNet) to preserve and reuse features
from earlier layers.
To understand how to arrive at it, let's take a small detour to mathematics.

In [ ]:
import autograd.numpy as np
from divelet_ml import nn

import divelet_ml as d2l

## From ResNet to DenseNet

Recall the Taylor expansion for functions. At the point $x = 0$ it can be written as

$$f(x) = f(0) + x \cdot \left[f'(0) + x \cdot \left[\frac{f''(0)}{2!}  + x \cdot \left[\frac{f'''(0)}{3!}  + \cdots \right]\right]\right]. \tag{8.7.1}$$


The key point is that it decomposes a function into terms of increasingly higher order. In a similar vein, ResNet decomposes functions into

$$f(\mathbf{x}) = \mathbf{x} + g(\mathbf{x}). \tag{8.7.2}$$

That is, ResNet decomposes $f$ into a simple linear term and a more complex
nonlinear one.
What if we wanted to capture (not necessarily add) information beyond two terms?
One such solution is DenseNet (Huang et al., 2017).

The main difference between ResNet (left) and DenseNet (right) in cross-layer connections: use of addition and use of concatenation. 

As shown in [Fig. 8.7.1](https://d2l.ai/chapter_convolutional-modern/densenet.html#fig-densenet-block), the key difference between ResNet and DenseNet is that in the latter case outputs are *concatenated* (denoted by $[,]$) rather than added.
As a result, we perform a mapping from $\mathbf{x}$ to its values after applying an increasingly complex sequence of functions:

$$\mathbf{x} \to \left[
\mathbf{x},
f_1(\mathbf{x}),
f_2\left(\left[\mathbf{x}, f_1\left(\mathbf{x}\right)\right]\right), f_3\left(\left[\mathbf{x}, f_1\left(\mathbf{x}\right), f_2\left(\left[\mathbf{x}, f_1\left(\mathbf{x}\right)\right]\right)\right]\right), \ldots\right]. \tag{8.7.3}$$

In the end, all these functions are combined in MLP to reduce the number of features again. In terms of implementation this is quite simple:
rather than adding terms, we concatenate them. The name DenseNet arises from the fact that the dependency graph between variables becomes quite dense. The final layer of such a chain is densely connected to all previous layers. The dense connections are shown in [Fig. 8.7.2](https://d2l.ai/chapter_convolutional-modern/densenet.html#fig-densenet).

Dense connections in DenseNet. Note how the dimensionality increases with depth.

The main components that comprise a DenseNet are *dense blocks* and *transition layers*. The former define how the inputs and outputs are concatenated, while the latter control the number of channels so that it is not too large, 
since the expansion $\mathbf{x} \to \left[\mathbf{x}, f_1(\mathbf{x}),
f_2\left(\left[\mathbf{x}, f_1\left(\mathbf{x}\right)\right]\right), \ldots \right]$ can be quite high-dimensional.


## Dense Blocks

DenseNet uses the modified "batch normalization, activation, and convolution"
structure of ResNet (see the exercise in [Section 8.6](https://d2l.ai/chapter_convolutional-modern/resnet.html#sec-resnet)).
First, we implement this convolution block structure.

In [ ]:
def conv_block(num_channels):
    return nn.Sequential(
        nn.LazyBatchNorm2d(), nn.ReLU(),
        nn.LazyConv2d(num_channels, kernel_size=3, padding=1))

A *dense block* consists of multiple convolution blocks, each using the same number of output channels. In the forward propagation, however, we concatenate the input and output of each convolution block on the channel dimension. Lazy evaluation allows us to adjust the dimensionality automatically.

In [ ]:
class DenseBlock(nn.Module):
    def __init__(self, num_convs, num_channels):
        layer = []
        for i in range(num_convs):
            layer.append(conv_block(num_channels))
        self.net = nn.Sequential(*layer)

    def forward(self, X):
        for blk in self.net:
            Y = blk(X)
            # Concatenate input and output of each block along the channels
            X = np.concatenate((X, Y), axis=1)
        return X

In the following example,
we define a `DenseBlock` instance with two convolution blocks of 10 output channels.
When using an input with three channels, we will get an output with  $3 + 10 + 10=23$ channels. The number of convolution block channels controls the growth in the number of output channels relative to the number of input channels. This is also referred to as the *growth rate*.

In [ ]:
blk = DenseBlock(2, 10)
X = np.random.randn(4, 3, 8, 8).astype(np.float32)
Y = blk(X)
Y.shape

## Transition Layers

Since each dense block will increase the number of channels, adding too many of them will lead to an excessively complex model. A *transition layer* is used to control the complexity of the model. It reduces the number of channels by using a $1\times 1$ convolution. Moreover, it halves the height and width via average pooling with a stride of 2.

In [ ]:
def transition_block(num_channels):
    return nn.Sequential(
        nn.LazyBatchNorm2d(), nn.ReLU(),
        nn.LazyConv2d(num_channels, kernel_size=1),
        nn.AvgPool2d(kernel_size=2, stride=2))

Apply a transition layer with 10 channels to the output of the dense block in the previous example.  This reduces the number of output channels to 10, and halves the height and width.

In [ ]:
blk = transition_block(10)
blk(Y).shape

## DenseNet Model

Next, we will construct a DenseNet model. DenseNet first uses the same single convolutional layer and max-pooling layer as in ResNet.

In [ ]:
class DenseNet(d2l.Classifier):
    def b1(self):
        return nn.Sequential(
            nn.LazyConv2d(64, kernel_size=7, stride=2, padding=3),
            nn.LazyBatchNorm2d(), nn.ReLU(),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1))

Then, similar to the four modules made up of residual blocks that ResNet uses,
DenseNet uses four dense blocks.
As with ResNet, we can set the number of convolutional layers used in each dense block. Here, we set it to 4, consistent with the ResNet-18 model in [Section 8.6](https://d2l.ai/chapter_convolutional-modern/resnet.html#sec-resnet). Furthermore, we set the number of channels (i.e., growth rate) for the convolutional layers in the dense block to 32, so 128 channels will be added to each dense block.

In ResNet, the height and width are reduced between each module by a residual block with a stride of 2. Here, we use the transition layer to halve the height and width and halve the number of channels. Similar to ResNet, a global pooling layer and a fully connected layer are connected at the end to produce the output.

In [ ]:
@d2l.add_to_class(DenseNet)
def __init__(self, num_channels=64, growth_rate=32, arch=(4, 4, 4, 4),
             lr=0.1, num_classes=10):
    super(DenseNet, self).__init__()
    self.save_hyperparameters()
    self.net = nn.Sequential(self.b1())
    for i, num_convs in enumerate(arch):
        self.net.add_module(f'dense_blk{i+1}', DenseBlock(num_convs,
                                                          growth_rate))
        # The number of output channels in the previous dense block
        num_channels += num_convs * growth_rate
        # A transition layer that halves the number of channels is added
        # between the dense blocks
        if i != len(arch) - 1:
            num_channels //= 2
            self.net.add_module(f'tran_blk{i+1}', transition_block(
                num_channels))
    self.net.add_module('last', nn.Sequential(
        nn.LazyBatchNorm2d(), nn.ReLU(),
        nn.AdaptiveAvgPool2d((1, 1)), nn.Flatten(),
        nn.LazyLinear(num_classes)))
    self.net.apply(d2l.init_cnn)

## Training

Since we are using a deeper network here, in this section, we will reduce the input height and width from 224 to 96 to simplify the computation.

<img src="../tablet.svg" alt="divelet-ml" title="Added by divelet-ml, not in the book"> **Not run in the browser.** In the book, the next cell trains DenseNet for 10 epochs on Fashion-MNIST resized to 96 × 96. This notebook stops here: counting the multiply-adds of the layers above gives about 129 million per image, so one epoch of 60,000 images would take about 4 hours in Pyodide (an estimate from the layer shapes and Pyodide's measured speed of about 1.5 billion multiply-adds per second; see `spikes/js-conv/count_macs.py`), and the resized images alone would take about 2.2 GB of memory. The book's cell is shown for reference; it needs a much faster machine than a browser tab.

```python
model = DenseNet(lr=0.01)
trainer = d2l.Trainer(max_epochs=10)
data = d2l.FashionMNIST(batch_size=128, resize=(96, 96))
trainer.fit(model, data)
```

## Summary and Discussion

The main components that comprise DenseNet are dense blocks and transition layers. For the latter, we need to keep the dimensionality under control when composing the network by adding transition layers that shrink the number of channels again.
In terms of cross-layer connections, in contrast to ResNet, where inputs and outputs are added together, DenseNet concatenates inputs and outputs on the channel dimension.
Although these concatenation operations
reuse features to achieve computational efficiency,
unfortunately they lead to heavy GPU memory consumption.
As a result,
applying DenseNet may require more memory-efficient implementations that may increase training time (Pleiss et al., 2017).


## Exercises

1. Why do we use average pooling rather than max-pooling in the transition layer?
1. One of the advantages mentioned in the DenseNet paper is that its model parameters are smaller than those of ResNet. Why is this the case?
1. One problem for which DenseNet has been criticized is its high memory consumption.
    1. Is this really the case? Try to change the input shape to $224\times 224$ to compare the actual GPU memory consumption empirically.
    1. Can you think of an alternative means of reducing the memory consumption? How would you need to change the framework?
1. Implement the various DenseNet versions presented in Table 1 of the DenseNet paper (Huang et al., 2017).
1. Design an MLP-based model by applying the DenseNet idea. Apply it to the housing price prediction task in [Section 5.7](https://d2l.ai/chapter_multilayer-perceptrons/kaggle-house-price.html#sec-kaggle-house).

Discussions of the original section: [book's forum](https://discuss.d2l.ai/t/88)

Previous: [Section 8.6: Residual Networks (ResNet) and ResNeXt](08.06_resnet.ipynb) | Next: [Section 8.8: Designing Convolution Network Architectures](08.08_cnn-design.ipynb)

## Attribution and changes

Adapted from *Dive into Deep Learning*, section
[Densely Connected Networks (DenseNet)](https://d2l.ai/chapter_convolutional-modern/densenet.html)
(source: [`chapter_convolutional-modern/densenet.md`](https://github.com/d2l-ai/d2l-en/blob/23d7a5aecceee57d1292c56e90cce307f183bb0a/chapter_convolutional-modern/densenet.md)
at commit `23d7a5aeccee`), by Aston Zhang, Zachary C. Lipton, Mu Li, Alexander J. Smola and contributors.
Text and figures: [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/). Code: adapted from the book's sample code, MIT-0.
Not affiliated with or endorsed by the book's authors.

**Changes from the original** (PyTorch edition):
- Code uses NumPy instead of PyTorch, so it runs in JupyterLite (Pyodide and xeus-python).
- `d2l` is divelet-ml's package (`import divelet_ml as d2l`) instead of the book's `d2l.torch`. `%matplotlib inline` is removed: both JupyterLite kernels display plots inline.
- The model is built from divelet-ml's `nn` module (`divelet_ml.nn`): NumPy stand-ins for the layers of `torch.nn`, whose lazy layers take their input sizes from the first batch. Weights are initialized by `d2l.init_cnn` after the first forward pass, as in the book. `torch.cat` becomes `np.concatenate`, and `torch.randn` `np.random.randn`, cast to `float32`. `stride` and `padding` of `nn.LazyConv2d` are passed by keyword, as divelet-ml's convolutional layers take `padding` before `stride`, the other way round from PyTorch's.
- The training cell is not run: a browser cannot train this model on the resized Fashion-MNIST images in a reasonable time or memory, so a note after the layer shapes says what the book does and why the notebook stops, and shows the book's cell for reference (the numbers come from counting the model's multiply-adds, `spikes/js-conv/count_macs.py`).
- Equations keep the book's numbers. Cross-references to other sections link to the book's site, and citations are written inline (author and year); the book's bibliography lists them.
- The d2lbook framework tabs for MXNet, TensorFlow and JAX are removed.